Use Pandas to read spam_sms data

In [5]:
import pandas as pd

df = pd.read_csv('spam_sms.csv')
print(df.head())

  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...


use NLTK for preprocessing

In [6]:
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords') # download stopwords
stop_words = set(stopwords.words('english')) #assign english stopwords only to stop_words

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\akraj\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Preprocess message by removing punctuations and stopwords also convert text to lower case

In [7]:
import string
def preprocess(text):
    text = text.lower()  # lowercase
    text = ''.join([ch for ch in text if ch not in string.punctuation])  # remove punctuation
    tokens = text.split()
    tokens = [word for word in tokens if word not in stop_words]  # remove stopwords
    return ' '.join(tokens)

df['clean_message'] = df['message'].apply(preprocess)

Use TfidfVectorizer to convert messages to numerical form

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df['clean_message'])

Use LabelEncoder to encode Labels (ham=0, spam=1)

In [9]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
Y = label_encoder.fit_transform(df['label'])

Use 20% of the labels

In [10]:
import numpy as np 

rng = np.random.RandomState(42)
n_total = len(Y)
n_labeled = int(0.2 * n_total)

Mask 80% as unlabeled

In [11]:
indices = np.arange(n_total)
rng.shuffle(indices)
y_train = np.copy(Y)
y_train[indices[n_labeled:]] = -1 # -1 means unlabeled for semi-supervised learners

Use LabelSpreading to apply knn model for prediction for semisupervised Learning

In [12]:
from sklearn.semi_supervised import LabelSpreading
model = LabelSpreading(kernel='knn', n_neighbors=5)
model.fit(X, y_train)

LabelSpreading(kernel='knn', n_neighbors=5)

Predict all labels (including previously unlabeled ones)

In [13]:
y_pred = model.transduction_

find actual values and predicted values of data

In [14]:
from sklearn.metrics import classification_report, accuracy_score

true_labels = Y[indices[n_labeled:]]
predicted_labels = y_pred[indices[n_labeled:]]

Accuracy score

In [15]:
print("Accuracy: ", accuracy_score(true_labels, predicted_labels))

Accuracy:  0.9073575594436967


Classification report

In [16]:
print("Classification report: ", classification_report(true_labels, predicted_labels, target_names=label_encoder.classes_))

Classification report:                precision    recall  f1-score   support

         ham       0.90      1.00      0.95      3860
        spam       1.00      0.31      0.47       598

    accuracy                           0.91      4458
   macro avg       0.95      0.65      0.71      4458
weighted avg       0.92      0.91      0.89      4458

